# IPL Win Predictor — Jupyter Version (self-trained)



In [1]:
# 1. Install dependencies (run once)
!pip install pandas numpy scikit-learn ipywidgets --quiet

In [2]:
# 2. Imports
import random
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
# 3. Load raw data
delivery = pd.read_csv('deliveries.csv')
match = pd.read_csv('matches.csv')
print(match.shape, delivery.shape)

(756, 18) (179078, 21)


In [4]:
# 4. Build match-level totals (first innings score = target)
total_score_df = delivery.groupby(['match_id', 'inning']).sum()['total_runs'].reset_index()
total_score_df = total_score_df[total_score_df['inning'] == 1]

match_df = match.merge(total_score_df[['match_id', 'total_runs']], left_on='id', right_on='match_id')

In [5]:
# 5. Keep only the 8 teams the app supports, normalize renamed franchises
teams = [
    'Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bangalore',
    'Kolkata Knight Riders', 'Kings XI Punjab', 'Chennai Super Kings',
    'Rajasthan Royals', 'Delhi Capitals'
]

match_df['team1'] = match_df['team1'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
match_df['team2'] = match_df['team2'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
match_df['team1'] = match_df['team1'].str.replace('Delhi Daredevils', 'Delhi Capitals')
match_df['team2'] = match_df['team2'].str.replace('Delhi Daredevils', 'Delhi Capitals')

match_df = match_df[match_df['team1'].isin(teams)]
match_df = match_df[match_df['team2'].isin(teams)]
match_df = match_df[match_df['dl_applied'] == 0]

match_df = match_df[['match_id', 'city', 'winner', 'total_runs']]

In [6]:
# 6. Build ball-by-ball second-innings features
delivery_df = match_df.merge(delivery, on='match_id')
delivery_df = delivery_df[delivery_df['inning'] == 2]

delivery_df['current_score'] = delivery_df.groupby('match_id')['total_runs_y'].cumsum()
delivery_df['runs_left'] = delivery_df['total_runs_x'] - delivery_df['current_score'] + 1
delivery_df['balls_left'] = 126 - (delivery_df['over'] * 6 + delivery_df['ball'])

delivery_df['player_dismissed'] = delivery_df['player_dismissed'].fillna(0)
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].apply(lambda x: x if x == 0 else 1)
wickets = delivery_df.groupby('match_id')['player_dismissed'].cumsum().values
delivery_df['wickets'] = 10 - wickets

delivery_df['crr'] = delivery_df.current_score * 6 / (120 - delivery_df.balls_left)
delivery_df['rrr'] = delivery_df.runs_left * 6 / delivery_df.balls_left

delivery_df = delivery_df.reset_index(drop=True)

In [7]:
# 7. Label the winner from the batting team's perspective
def is_win(df):
    return [1 if row.winner == row.batting_team else 0 for _, row in df.iterrows()]

delivery_df['winner'] = is_win(delivery_df)

final_df = delivery_df[['match_id', 'batting_team', 'bowling_team', 'city',
                         'runs_left', 'balls_left', 'wickets', 'total_runs_x',
                         'crr', 'rrr', 'winner']]
final_df = final_df.sample(final_df.shape[0], random_state=1)

final_df['batting_team'] = final_df['batting_team'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
final_df['bowling_team'] = final_df['bowling_team'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
final_df['batting_team'] = final_df['batting_team'].str.replace('Delhi Daredevils', 'Delhi Capitals')
final_df['bowling_team'] = final_df['bowling_team'].str.replace('Delhi Daredevils', 'Delhi Capitals')

In [8]:
# 8. Fill missing city with a plausible home city, then clean up
cities_dict = {
    'Royal Challengers Bangalore': 'Bengaluru',
    'Chennai Super Kings': 'Chennai',
    'Kings XI Punjab': 'Mumbai',
    'Kolkata Knight Riders': 'Kolkata',
    'Delhi Capitals': 'Delhi',
    'Rajasthan Royals': 'Jaipur',
    'Mumbai Indians': 'Mumbai',
    'Sunrisers Hyderabad': 'Hyderabad',
}

final_df['city'] = final_df['city'].fillna(0)

def fill_city(x):
    if x.city == 0:
        team = [x.batting_team, x.bowling_team][random.randint(0, 1)]
        return cities_dict[team]
    return x.city

final_df['city'] = final_df.apply(fill_city, axis=1)

final_df.dropna(inplace=True)
final_df = final_df[final_df.balls_left != 0]

print(final_df.shape)
final_df.head()

(72172, 11)


,match_id,batting_team,bowling_team,city,runs_left,balls_left,wickets,total_runs_x,crr,rrr,winner
69323,11324,Sunrisers Hyderabad,Chennai Super Kings,Hyderabad,35,48,8,139,8.750000,4.375000,1
38573,421,Kings XI Punjab,Mumbai Indians,Mumbai,54,27,3,174,7.806452,12.000000,0
3971,65,Rajasthan Royals,Kings XI Punjab,Jaipur,53,43,6,166,8.883117,7.395349,1
38311,418,Kolkata Knight Riders,Chennai Super Kings,Chennai,94,42,7,200,8.230769,13.428571,0
1153,18,Kolkata Knight Riders,Delhi Capitals,Delhi,17,12,5,168,8.444444,8.500000,1


In [9]:
# 9. Train the model
X = final_df.drop(columns=['winner', 'match_id'])
y = final_df['winner']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

trf = ColumnTransformer([
    ('trf', OneHotEncoder(drop='first', handle_unknown='ignore'), ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

pipe = Pipeline([
    ('step1', trf),
    ('step2', LogisticRegression(solver='liblinear'))
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, y_pred), 4))

Test accuracy: 0.8054


In [10]:
# 10. (Optional) Save the freshly-trained model so you don't have to retrain every time
import pickle
with open('pipe.pkl', 'wb') as f:
    pickle.dump(pipe, f)
print('Saved pipe.pkl trained with your current scikit-learn version.')

Saved pipe.pkl trained with your current scikit-learn version.


## Interactive predictor
Run the two cells below to get the same dropdown/button UI as `app.py`.

In [11]:
# 11. Build the interactive widgets
cities = sorted(final_df['city'].unique().tolist())

batting_team_w = widgets.Dropdown(options=teams, description='Batting team:', style={'description_width': 'initial'})
bowling_team_w = widgets.Dropdown(options=teams, value=teams[1], description='Bowling team:', style={'description_width': 'initial'})
city_w = widgets.Dropdown(options=cities, description='City:', style={'description_width': 'initial'})

target_w = widgets.BoundedIntText(value=150, min=0, max=300, description='Target:', style={'description_width': 'initial'})
score_w = widgets.BoundedIntText(value=0, min=0, max=300, description='Score:', style={'description_width': 'initial'})
wickets_w = widgets.BoundedIntText(value=0, min=0, max=9, description='Wickets lost:', style={'description_width': 'initial'})
overs_w = widgets.BoundedFloatText(value=1.0, min=0.1, max=20, step=0.1, description='Overs completed:', style={'description_width': 'initial'})

predict_btn = widgets.Button(description='Predict Probability', button_style='success')
output = widgets.Output()

def on_predict_clicked(b):
    with output:
        clear_output()
        target = target_w.value
        score = score_w.value
        wk = wickets_w.value
        overs = overs_w.value

        if overs <= 0:
            print("Overs completed must be greater than 0.")
            return
        if wk > 10:
            print("Please check your inputs.")
            return

        runs_left = target - score
        balls_left = 120 - int(overs * 6)
        wickets_left = 10 - wk
        crr = score / overs
        rrr = (runs_left * 6 / balls_left) if balls_left > 0 else 0

        df = pd.DataFrame({
            'batting_team': [batting_team_w.value],
            'bowling_team': [bowling_team_w.value],
            'city': [city_w.value],
            'runs_left': [runs_left],
            'balls_left': [balls_left],
            'wickets': [wickets_left],
            'total_runs_x': [target],
            'crr': [crr],
            'rrr': [rrr]
        })

        result = pipe.predict_proba(df)
        r_1 = round(result[0][0] * 100)
        r_2 = round(result[0][1] * 100)

        print("Winning Probability")
        print(f"{batting_team_w.value}: {r_2}%")
        print(f"{bowling_team_w.value}: {r_1}%")

predict_btn.on_click(on_predict_clicked)

In [12]:
# 12. Display the app
display(widgets.HTML("<h2>IPL Win Predictor</h2>"))
display(widgets.HBox([batting_team_w, bowling_team_w]))
display(city_w)
display(target_w)
display(widgets.HBox([score_w, wickets_w, overs_w]))
display(predict_btn, output)

HTML(value='<h2>IPL Win Predictor</h2>')

Dropdown(description='City:', options=('Abu Dhabi', 'Ahmedabad', 'Bangalore', 'Bengaluru', 'Bloemfontein', 'Ca…

BoundedIntText(value=150, description='Target:', max=300, style=DescriptionStyle(description_width='initial'))

Button(button_style='success', description='Predict Probability', style=ButtonStyle())

Output()